<a href="https://colab.research.google.com/github/20deepak28/demand-analysis-project/blob/main/deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


**MOUNTING AT Google Drive, for importing our project**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive"))

['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [ ]:
import os

root = "/content/drive/MyDrive"

for item in os.listdir(root):
    print(item)

Colab Notebooks
Retail-Demand-Forecasting


In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Retail-Demand-Forecasting"

print(os.listdir(PROJECT_PATH))

['Data', 'models', 'notebooks', 'dashboard', 'README.md', 'reports']


In [ ]:
import os

FEATURE_PATH = (
    PROJECT_PATH
    + "/Data/processed/full_dataset/featured"
)

files = sorted(os.listdir(FEATURE_PATH))

print("Total files:", len(files))
print("First 5:", files[:5])

Total files: 31
First 5: ['sales_chunk_000.parquet', 'sales_chunk_001.parquet', 'sales_chunk_002.parquet', 'sales_chunk_003.parquet', 'sales_chunk_004.parquet']


### LOAD the dataset and inspect its structure

In [ ]:
import pandas as pd
from pathlib import Path

FEATURE_PATH = Path(PROJECT_PATH) / "Data/processed/full_dataset/featured"

parquet_files = sorted(FEATURE_PATH.glob("*.parquet"))

df = pd.read_parquet(parquet_files[0])

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Shape: (1941000, 27)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'weekday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'wm_yr_wk', 'sell_price', 'revenue', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'is_event']
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales       date    weekday  ...  wm_yr_wk  sell_price  \
0       CA  d_1      0 2011-01-29   Saturday  ...     11101         NaN   
1       CA  d_2      0 2011-01-3

### Check data types and missing values

In [ ]:
print(df[["sales", "lag_1", "lag_7", "lag_28",
          "rolling_mean_7", "rolling_mean_28"]].isnull().sum())

print("\nData types:")
print(df[["sales", "lag_1", "lag_7", "lag_28"]].dtypes)

sales                  0
lag_1               1000
lag_7               7000
lag_28             28000
rolling_mean_7      7000
rolling_mean_28    28000
dtype: int64

Data types:
sales       int16
lag_1     float64
lag_7     float64
lag_28    float64
dtype: object


In [ ]:
print("Date range:", df["date"].min(), "to", df["date"].max())

print("Unique product-store series:", df["id"].nunique())

print("\nRows per series:")
print(df.groupby("id").size().describe())

Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00
Unique product-store series: 1000

Rows per series:
count    1000.0
mean     1941.0
std         0.0
min      1941.0
25%      1941.0
50%      1941.0
75%      1941.0
max      1941.0
dtype: float64


### Prepare the first chunk for LSTM

In [ ]:
import numpy as np
import pandas as pd

# Sort data chronologically within each product-store series
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# Convert sales to float32 and apply log transformation
df["sales_log"] = np.log1p(df["sales"].astype(np.float32))

# Check
print("Shape:", df.shape)
print("\nData types:")
print(df[["sales", "sales_log"]].dtypes)

print("\nSample:")
display(df[["id", "date", "sales", "sales_log"]].head())

Shape: (1941000, 28)

Data types:
sales          int16
sales_log    float32
dtype: object

Sample:


,id,date,sales,sales_log
0,HOBBIES_1_001_CA_1_evaluation,2011-01-29,0,0.0
1,HOBBIES_1_001_CA_1_evaluation,2011-01-30,0,0.0
2,HOBBIES_1_001_CA_1_evaluation,2011-01-31,0,0.0
3,HOBBIES_1_001_CA_1_evaluation,2011-02-01,0,0.0
4,HOBBIES_1_001_CA_1_evaluation,2011-02-02,0,0.0


### Identify our LSTM feautes

In [ ]:
print("Available columns:")
print(df.columns.tolist())

print("\nSelected feature information:")
features = [
    "sales",
    "sell_price",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "is_event",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

print(df[features].dtypes)
print("\nMissing values:")
print(df[features].isnull().sum())

Available columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'weekday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'wm_yr_wk', 'sell_price', 'revenue', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'is_event', 'sales_log']

Selected feature information:
sales                int16
sell_price         float32
lag_1              float64
lag_7              float64
lag_28             float64
rolling_mean_7     float64
rolling_mean_28    float64
day_of_week          int64
month                int64
is_event              int8
snap_CA              int64
snap_TX              int64
snap_WI              int64
dtype: object

Missing values:
sales                   0
sell_price         392504
lag_1                1000
lag_7                7000
lag_28              28000
rolling_mean_7       7000
rolling_mean_28     28000
day_of_week             0
month                   0
is_event      

### Investigate missing prices

In [ ]:
missing_price = df[df["sell_price"].isna()]

print("Missing price rows:", len(missing_price))

print("\nSales distribution where price is missing:")
print(missing_price["sales"].value_counts().head(10))

print("\nPositive sales with missing price:")
print((missing_price["sales"] > 0).sum())

print("\nMissing price percentage:")
print(round(len(missing_price) / len(df) * 100, 2), "%")

Missing price rows: 392504

Sales distribution where price is missing:
sales
0    392504
Name: count, dtype: int64

Positive sales with missing price:
0

Missing price percentage:
20.22 %


### This confirms that in this chunk, all missing-price observations have zero sales.

In [ ]:
# Identify missing prices
df["price_missing"] = df["sell_price"].isna().astype("int8")

# Create a separate model-ready price column
df["sell_price_filled"] = df["sell_price"].fillna(0)

print("Missing-price indicator:")
print(df["price_missing"].value_counts())

print("\nRemaining missing prices:")
print(df["sell_price_filled"].isna().sum())

print("\nPositive sales with missing price:")
print(((df["price_missing"] == 1) & (df["sales"] > 0)).sum())

Missing-price indicator:
price_missing
0    1548496
1     392504
Name: count, dtype: int64

Remaining missing prices:
0

Positive sales with missing price:
0


### SELECT LSTM input featurs

In [ ]:
FEATURES = [
    "sales_log",
    "sell_price_filled",
    "price_missing",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "is_event",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

TARGET = "sales_log"

print("Total input features:", len(FEATURES))
print(FEATURES)

Total input features: 14
['sales_log', 'sell_price_filled', 'price_missing', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'day_of_week', 'month', 'is_event', 'snap_CA', 'snap_TX', 'snap_WI']


### Convert features to float 32

In [ ]:
# Convert model features to memory-efficient float32
df[FEATURES] = df[FEATURES].astype(np.float32)

print(df[FEATURES].dtypes)
print("\nMissing values:")
print(df[FEATURES].isna().sum())

sales_log            float32
sell_price_filled    float32
price_missing        float32
lag_1                float32
lag_7                float32
lag_28               float32
rolling_mean_7       float32
rolling_mean_28      float32
day_of_week          float32
month                float32
is_event             float32
snap_CA              float32
snap_TX              float32
snap_WI              float32
dtype: object

Missing values:
sales_log                0
sell_price_filled        0
price_missing            0
lag_1                 1000
lag_7                 7000
lag_28               28000
rolling_mean_7        7000
rolling_mean_28      28000
day_of_week              0
month                    0
is_event                 0
snap_CA                  0
snap_TX                  0
snap_WI                  0
dtype: int64


### Check Missing Values in alll Features

In [ ]:
print("Total missing values:", df[FEATURES].isna().sum().sum())

print("\nMissing values by feature:")
print(df[FEATURES].isna().sum())

Total missing values: 71000

Missing values by feature:
sales_log                0
sell_price_filled        0
price_missing            0
lag_1                 1000
lag_7                 7000
lag_28               28000
rolling_mean_7        7000
rolling_mean_28      28000
day_of_week              0
month                    0
is_event                 0
snap_CA                  0
snap_TX                  0
snap_WI                  0
dtype: int64


### HAndle missing lag features

In [ ]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

df[LAG_FEATURES] = df[LAG_FEATURES].fillna(0)

print("Remaining missing values:", df[FEATURES].isna().sum().sum())

print("Dataset shape:", df.shape)

Remaining missing values: 0
Dataset shape: (1941000, 30)


In [ ]:
print(df[FEATURES].describe().T)

                       count      mean       std  min   25%       50%  \
sales_log          1941000.0  0.374363  0.613324  0.0  0.00  0.000000   
sell_price_filled  1941000.0  4.166167  4.402140  0.0  0.72  2.970000   
price_missing      1941000.0  0.202217  0.401569  0.0  0.00  0.000000   
lag_1              1941000.0  0.922700  2.541588  0.0  0.00  0.000000   
lag_7              1941000.0  0.919489  2.539431  0.0  0.00  0.000000   
lag_28             1941000.0  0.906791  2.531924  0.0  0.00  0.000000   
rolling_mean_7     1941000.0  0.919882  1.808067  0.0  0.00  0.285714   
rolling_mean_28    1941000.0  0.909908  1.681993  0.0  0.00  0.357143   
day_of_week        1941000.0  3.002576  2.000644  0.0  1.00  3.000000   
month              1941000.0  6.334879  3.437055  1.0  3.00  6.000000   
is_event           1941000.0  0.081401  0.273475  0.0  0.00  0.000000   
snap_CA            1941000.0  0.329727  0.465153  0.0  0.00  0.000000   
snap_TX            1941000.0  0.329727  0.465153  0

### Define training dates

In [ ]:
TRAIN_END = pd.Timestamp("2015-04-07")
VALID_START = pd.Timestamp("2015-04-08")

print("Training end:", TRAIN_END)
print("Validation start:", VALID_START)

Training end: 2015-04-07 00:00:00
Validation start: 2015-04-08 00:00:00


### Initialize Scalar

In [ ]:
from sklearn.preprocessing import StandardScaler

SCALED_FEATURES = [
    "sales_log",
    "sell_price_filled",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month"
]

scaler = StandardScaler()

print("Scaler initialized.")
print("Features to scale:", len(SCALED_FEATURES))

Scaler initialized.
Features to scale: 9


### Fit the common scaler

In [ ]:
from pathlib import Path

scaler = StandardScaler()

for i, file in enumerate(parquet_files, 1):

    temp = pd.read_parquet(
        file,
        columns=["date", "sales", "sell_price",
                 "lag_1", "lag_7", "lag_28",
                 "rolling_mean_7", "rolling_mean_28",
                 "day_of_week", "month"]
    )

    temp["date"] = pd.to_datetime(temp["date"])

    # Prepare features
    temp["sales_log"] = np.log1p(
        temp["sales"].astype(np.float32)
    )

    temp["sell_price_filled"] = temp["sell_price"].fillna(0)

    temp[SCALED_FEATURES] = (
        temp[SCALED_FEATURES].fillna(0).astype(np.float32)
    )

    # Training period only
    train_data = temp.loc[
        temp["date"] <= TRAIN_END, SCALED_FEATURES
    ]

    scaler.partial_fit(train_data)

    print(f"Processed chunk {i}/31")

print("\nCommon scaler fitted successfully!")

Processed chunk 1/31
Processed chunk 2/31
Processed chunk 3/31
Processed chunk 4/31
Processed chunk 5/31
Processed chunk 6/31
Processed chunk 7/31
Processed chunk 8/31
Processed chunk 9/31
Processed chunk 10/31
Processed chunk 11/31
Processed chunk 12/31
Processed chunk 13/31
Processed chunk 14/31
Processed chunk 15/31
Processed chunk 16/31
Processed chunk 17/31
Processed chunk 18/31
Processed chunk 19/31
Processed chunk 20/31
Processed chunk 21/31
Processed chunk 22/31
Processed chunk 23/31
Processed chunk 24/31
Processed chunk 25/31
Processed chunk 26/31
Processed chunk 27/31
Processed chunk 28/31
Processed chunk 29/31
Processed chunk 30/31
Processed chunk 31/31

Common scaler fitted successfully!


### Verify scaling statistics

In [ ]:
print("Feature means:")
print(scaler.mean_)

print("\nFeature standard deviations:")
print(scaler.scale_)

print("\nNumber of features:", scaler.n_features_in_)

Feature means:
[0.36777847 3.23104287 1.08506432 1.07993556 1.06400832 1.08063139
 1.06646925 3.         6.34640523]

Feature standard deviations:
[0.66714761 3.46785279 3.93563545 3.92929639 3.91159486 3.44780093
 3.2816961  2.00163332 3.46998382]

Number of features: 9


### Save the scaler

In [ ]:
import joblib

MODEL_PATH = Path(PROJECT_PATH) / "models"
MODEL_PATH.mkdir(parents=True, exist_ok=True)

scaler_path = MODEL_PATH / "lstm_feature_scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Scaler saved:", scaler_path)

Scaler saved: /content/drive/MyDrive/Retail-Demand-Forecasting/models/lstm_feature_scaler.pkl


### PRepare the sequence generator

In [ ]:
SEQ_LENGTH = 28

BINARY_FEATURES = [
    "price_missing",
    "is_event",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

MODEL_FEATURES = SCALED_FEATURES + BINARY_FEATURES

print("Total model features:", len(MODEL_FEATURES))
print("Sequence length:", SEQ_LENGTH)

Total model features: 14
Sequence length: 28


### Prepare the first parquet chunk

In [ ]:
def prepare_chunk(file):

    temp = pd.read_parquet(file)

    temp["date"] = pd.to_datetime(temp["date"])

    # Target transformation
    temp["sales_log"] = np.log1p(
        temp["sales"].astype(np.float32)
    )

    # Missing price handling
    temp["price_missing"] = (
        temp["sell_price"].isna().astype(np.int8)
    )

    temp["sell_price_filled"] = (
        temp["sell_price"].fillna(0)
    )

    # Handle initial lag NaNs
    temp[SCALED_FEATURES] = (
        temp[SCALED_FEATURES]
        .fillna(0)
        .astype(np.float32)
    )

    # Apply common scaler
    temp[SCALED_FEATURES] = scaler.transform(
        temp[SCALED_FEATURES]
    )

    # Preserve binary features
    temp[BINARY_FEATURES] = (
        temp[BINARY_FEATURES].astype(np.float32)
    )

    return temp

In [ ]:
test_chunk = prepare_chunk(parquet_files[0])

print("Shape:", test_chunk.shape)
print("Features:", test_chunk[MODEL_FEATURES].shape)
print("Missing values:", test_chunk[MODEL_FEATURES].isna().sum().sum())

Shape: (1941000, 30)
Features: (1941000, 14)
Missing values: 0


## Genrate Multivariate sequences

In [ ]:
def create_sequences(group):
    X = group[MODEL_FEATURES].to_numpy(dtype=np.float32)
    y = group["sales_log"].to_numpy(dtype=np.float32)
    dates = group["date"].to_numpy()

    X_seq = np.lib.stride_tricks.sliding_window_view(
        X, window_shape=SEQ_LENGTH, axis=0
    ).transpose(0, 2, 1)

    # Remove final window (no next-day target)
    X_seq = X_seq[:-1]

    y_seq = y[SEQ_LENGTH:]
    target_dates = dates[SEQ_LENGTH:]

    train_mask = target_dates <= np.datetime64(TRAIN_END)
    valid_mask = target_dates >= np.datetime64(VALID_START)

    return (
        X_seq[train_mask],
        y_seq[train_mask],
        X_seq[valid_mask],
        y_seq[valid_mask]
    )

In [ ]:
first_product = test_chunk[
    test_chunk["id"] == test_chunk["id"].iloc[0]
].sort_values("date").reset_index(drop=True)

X_train, y_train, X_valid, y_valid = create_sequences(first_product)

print("Training X:", X_train.shape)
print("Training y:", y_train.shape)

print("Validation X:", X_valid.shape)
print("Validation y:", y_valid.shape)

Training X: (1502, 28, 14)
Training y: (1502,)
Validation X: (411, 28, 14)
Validation y: (411,)


### verify no data leakage or missing value

In [ ]:
print("Training NaNs:", np.isnan(X_train).sum())
print("Validation NaNs:", np.isnan(X_valid).sum())

print("\nTraining target NaNs:", np.isnan(y_train).sum())
print("Validation target NaNs:", np.isnan(y_valid).sum())

print("\nTraining date range:",
      first_product["date"].iloc[28],
      "to", TRAIN_END)

print("Validation starts:", VALID_START)

Training NaNs: 0
Validation NaNs: 0

Training target NaNs: 0
Validation target NaNs: 0

Training date range: 2011-02-26 00:00:00 to 2015-04-07 00:00:00
Validation starts: 2015-04-08 00:00:00


### Cheching memory

In [ ]:
print("Training X memory:",
      round(X_train.nbytes / (1024**2), 2), "MB")

print("Validation X memory:",
      round(X_valid.nbytes / (1024**2), 2), "MB")

print("Training samples:", len(X_train))
print("Validation samples:", len(X_valid))

Training X memory: 2.25 MB
Validation X memory: 0.61 MB
Training samples: 1502
Validation samples: 411


In [ ]:
print("Total chunks:", len(parquet_files))

for i, file in enumerate(parquet_files[:3], 1):
    temp = pd.read_parquet(file, columns=["id"])
    print(
        f"Chunk {i}: {temp['id'].nunique()} product-store series"
    )
    del temp

Total chunks: 31
Chunk 1: 1000 product-store series
Chunk 2: 1000 product-store series
Chunk 3: 1000 product-store series


In [ ]:
last_chunk = pd.read_parquet(
    parquet_files[-1],
    columns=["id"]
)

print("Last chunk series:", last_chunk["id"].nunique())
print("Total rows:", len(last_chunk))

del last_chunk

Last chunk series: 490
Total rows: 951090


### create a streaming sequence generator

In [ ]:
def stream_sequences(parquet_files, batch_size=512):

    for file_idx, file in enumerate(parquet_files, 1):

        chunk = prepare_chunk(file)

        chunk = chunk.sort_values(
            ["id", "date"]
        )

        for product_id, group in chunk.groupby("id", sort=False):

            X_tr, y_tr, X_val, y_val = create_sequences(group)

            # Training batches
            for start in range(0, len(X_tr), batch_size):
                end = start + batch_size

                yield (
                    X_tr[start:end],
                    y_tr[start:end],
                    "train"
                )

            # Validation batches
            for start in range(0, len(X_val), batch_size):
                end = start + batch_size

                yield (
                    X_val[start:end],
                    y_val[start:end],
                    "valid"
                )

        print(f"Processed chunk {file_idx}/{len(parquet_files)}")

        del chunk

In [ ]:
generator = stream_sequences(parquet_files[:1], batch_size=512)

X_batch, y_batch, batch_type = next(generator)

print("Batch type:", batch_type)
print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)
print("X data type:", X_batch.dtype)
print("NaN values:", np.isnan(X_batch).sum())

Batch type: train
X shape: (512, 28, 14)
y shape: (512,)
X data type: float32
NaN values: 0


### TEST streaming output

In [ ]:
generator = stream_sequences(parquet_files[:1], batch_size=512)

X_batch, y_batch, batch_type = next(generator)

print("Batch type:", batch_type)
print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)
print("X data type:", X_batch.dtype)
print("NaN values:", np.isnan(X_batch).sum())

Batch type: train
X shape: (512, 28, 14)
y shape: (512,)
X data type: float32
NaN values: 0


### Create PyTorch batch conversion

In [ ]:
import torch

def to_torch_batch(X, y):
    X_tensor = torch.from_numpy(X).float()
    y_tensor = torch.from_numpy(y).float()

    return (
        X_tensor.to("cuda"),
        y_tensor.to("cuda")
    )

X_gpu, y_gpu = to_torch_batch(X_batch, y_batch)

print("X device:", X_gpu.device)
print("y device:", y_gpu.device)
print("X shape:", X_gpu.shape)
print("y shape:", y_gpu.shape)

X device: cuda:0
y device: cuda:0
X shape: torch.Size([512, 28, 14])
y shape: torch.Size([512])
